In [ ]:
Data Loader

In [ ]:
# Delete the "Autre Surface" patches

import os, glob

DATA_DIR = os.path.expanduser('~/Desktop/training_data/')

to_delete = glob.glob(os.path.join(DATA_DIR, '*_Autre surface*.tif'))
print(f"Found {len(to_delete)} 'Autre surface' patches to delete")

for f in to_delete:
    os.remove(f)

print("✅ Done!")

In [ ]:
import os
import re
import glob
from PIL import Image
from torch.utils.data import Dataset

class PrairieDataset(Dataset):

    # mapping between label class names and indices
    LABEL_CLASSES = {
        'Plus'  : 1,
        'Moins' : 0,
    }

    def __init__(self, data_dir, transform=None, split='train',
                 val_size=0.15, test_size=0.15):
        self.transform = transform
        self.data_dir  = data_dir

        # LOAD ALL PATCHES 
        all_patches = []
        for path in glob.glob(os.path.join(data_dir, '*.tif')):
            filename = os.path.basename(path)

            # parse filename: {x}_{y}_{qualite}.tif
            match = re.match(r'([\d.]+)_([\d.]+)_(.+)\.tif$', filename)
            if not match:
                continue

            x       = float(match.group(1))
            y       = float(match.group(2))
            qualite = match.group(3)
            
            if qualite not in self.LABEL_CLASSES:
                continue  # skips anything that's not Plus or Moins

            all_patches.append((path, x, y, self.LABEL_CLASSES[qualite]))

        # SPATIAL SPLIT by easting (x coordinate)
        all_patches.sort(key=lambda p: p[1])
        n         = len(all_patches)
        train_end = int(n * (1 - val_size - test_size))
        val_end   = int(n * (1 - test_size))

        if split == 'train':
            self.data = [(p[0], p[3]) for p in all_patches[:train_end]]
        elif split == 'val':
            self.data = [(p[0], p[3]) for p in all_patches[train_end:val_end]]
        elif split == 'test':
            self.data = [(p[0], p[3]) for p in all_patches[val_end:]]
        else:
            raise ValueError(f"Unknown split '{split}'. Use 'train', 'val', or 'test'.")

        print(f"Split '{split}': {len(self.data)} patches")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, x):
        img_path, label = self.data[x]

        img = Image.open(img_path).convert('RGB')

        if self.transform is not None:
            img = self.transform(img)

        return img, label

In [ ]:
# Check if East Split ok

import re, glob, os
import numpy as np

DATA_DIR = os.path.expanduser('~/Desktop/training_data/')

patches = []
for path in glob.glob(os.path.join(DATA_DIR, '*.tif')):
    filename = os.path.basename(path)
    match = re.match(r'([\d.]+)_([\d.]+)_(.+)\.tif$', filename)
    if not match:
        continue
    x       = float(match.group(1))
    qualite = match.group(3)
    if qualite in ('Plus', 'Moins'):
        patches.append((x, qualite))

patches.sort(key=lambda p: p[0])
n         = len(patches)
train_end = int(n * 0.8)
val_end   = int(n * 0.9)

splits = {
    'train' : patches[:train_end],
    'val'   : patches[train_end:val_end],
    'test'  : patches[val_end:]
}

print(f"{'Split':<8} {'Total':>8} {'Plus':>8} {'Moins':>8} {'Plus%':>8}")
print("─" * 45)
for split_name, split_patches in splits.items():
    total = len(split_patches)
    plus  = sum(1 for p in split_patches if p[1] == 'Plus')
    moins = total - plus
    print(f"{split_name:<8} {total:>8} {plus:>8} {moins:>8} {100*plus/total:>7.1f}%")